In [1]:
import sys
sys.path.insert(1, '../../code/')
import os
import pickle
import torch
import numpy as np
from tqdm import tqdm
from datetime import datetime
import tractogramReader as tr
import time
from matplotlib import pyplot as plt
from scipy import stats
import seaborn as sns
import random
import nibabel as nib
from model import ConvAE_256res
from utils import *
from clustering import Kmeans

In [ ]:
device = set_device()
RANDOM_SEED = 42
res = 256
data_shape = [3, res]
start = time.time()
LS = torch.load('HCP_Embeddings/HCP_38sub_N256_L1_64ld_latentSpace.pt', map_location=torch.device('cpu'))
LS = torch.reshape(LS,(LS.shape[0], LS.shape[1]))

In [ ]:
LS_keys = list(np.arange(LS.shape[0]))
LS_dict = {LS_keys[i]: LS[i] for i in tqdm(range(len(LS_keys)))}

In [ ]:
clusters_out_file = 'newClustering_centers'
LS_dict_update = LS_dict
clusters = {}
label = 0
while LS_dict_update:
    # get the existing keys of the updated list
    keys = list(LS_dict_update.keys())
    row_to_compare = LS[keys[0],:]
    dist = torch.norm(LS[keys,:]-row_to_compare, dim=1)
    
    # filtering the keys based on the distance threshold
    mask = dist < 200
    filtered_keys = np.array(keys)[mask]
        
    LS_dict_update = {key: value for key, value in LS_dict_update.items() if key not in filtered_keys}
    clusters[label] = filtered_keys
    label += 1
    
cluster_centers = np.zeros((len(clusters),64), np.float32)

for i in range(len(clusters)):
    cluster_centers[i]= LS[list(clusters[i])].mean(axis=0)
    
cluster_centers.astype(np.float32).tofile(clusters_out_file)